# 02. Текст: tokenizer, Qwen, CLIP и два refiner

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 3](https://diffusion.csail.mit.edu/2026/docs/20260123_Lecture_03.pdf) · [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** различать IDs, hidden states, pooled condition и padding mask; проверить crop и text-projection cache

**Перед началом:** K6 01; attention из Lecture 4. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/01_text_embedders.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §3.1–3.2](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/text_embedder.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/text_embedder.py)
- [`core/components/dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/dit.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','main'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '02_text_conditioning'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Четыре разных представления текста

Tokenizer возвращает целые IDs; embedding lookup превращает ID в вектор; Qwen contextualizes
вектор с учётом других токенов; CLIP даёт pooled representation. K6 использует Qwen states ширины 3584
и CLIP pooled 768. После crop доступно до 1024 токенов; CLIP ограничен 77.
Текстовый VLM используется как encoder. Для T2AV его image/video inputs пустые.

У frozen Qwen один набор весов, но DiT содержит **два собственных trainable refiner**:
`video_text_transformer_blocks` и `audio_text_transformer_blocks`. Выход общего encoder
не равен выходу этих refiner. `text_embedder.py` native пути передаёт Qwen attention mask.

Ниже искусственные IDs показывают только crop/padding. Это **не реализация Qwen tokenizer**:
мы не имитируем его BPE/byte vocabulary обычным split по пробелу.


In [ ]:
crop,max_length=129,8
ids=torch.arange(2*(crop+max_length)).reshape(2,crop+max_length)
valid_lengths=torch.tensor([crop+6,crop+3])
mask=torch.arange(crop+max_length)[None,:]<valid_lengths[:,None]
kept_ids=ids[:,crop:];kept_mask=mask[:,crop:]
assert kept_ids.shape==(2,8) and kept_mask.sum(1).tolist()==[6,3]
print('IDs after crop:',kept_ids);print('valid:',kept_mask)
fig,ax=plt.subplots(figsize=(7,2));ax.imshow(kept_mask,cmap='Blues',vmin=0,vmax=1)
ax.set_xlabel('Token after system-template crop');ax.set_yticks([0,1]);plt.show()
metrics['valid_tokens_after_crop']=kept_mask.sum(1).tolist()


## 2. Маска ограничивает доступ к K/V

В PyTorch SDPA boolean `True` означает разрешённый key. Мы изменим padded value на очень большое
число. С правильной маской результат не меняется. Это проверка контракта downstream attention,
не достаточная проверка tokenizer/template: реальный crop зависит от конкретных processor assets.


In [ ]:
q=torch.randn(2,2,3,4);k=torch.randn(2,2,8,4);v=torch.randn(2,2,8,4)
attention_mask=kept_mask[:,None,None,:]
a=F.scaled_dot_product_attention(q,k,v,attn_mask=attention_mask)
v2=v+((~kept_mask)[:,None,:,None])*1000
b=F.scaled_dot_product_attention(q,k,v2,attn_mask=attention_mask)
error=(a-b).abs().max().item();assert error==0
metrics['masked_value_perturbation_max_error']=error
print('Masked value perturbation error:',error)
show_source('core/components/text_embedder.py','encode_qwen')


## 3. Что можно кэшировать, а что зависит от времени

Native DiT кэширует time-independent token/pooled projections, затем на каждом шаге
прибавляет time embedding и выполняет time-modulated text blocks. Проверим hit по hooks.
Ключ использует data pointer и shape: нельзя менять cached tensor in-place и ожидать автоматической
инвалидации. Кэш очищают перед новой генерацией. В учебном training очищаем перед каждым forward,
чтобы не переиспользовать старый autograd graph.


In [ ]:
model=tiny_model().eval();_,_,kw=tiny_inputs(model)
hits=[]
hook=model.video_text_embeddings.register_forward_hook(lambda *args:hits.append(1))
with torch.no_grad():
    model.clear_text_proj_cache()
    p1=model._project_text_tokens('video',kw['text_embed'],kw['pooled_text_embed'])
    p2=model._project_text_tokens('video',kw['text_embed'],kw['pooled_text_embed'])
    time1=model._time_embed('video',torch.tensor([200.]),p1[1])
    time2=model._time_embed('video',torch.tensor([800.]),p2[1])
hook.remove()
assert len(hits)==1 and torch.equal(p1[0],p2[0])
metrics['projection_calls_two_reads']=len(hits)
metrics['time_condition_change']=(time1-time2).abs().max().item()
print('projection calls:',len(hits),'time delta:',metrics['time_condition_change'])


## Самостоятельное исследование

1. Уберите mask, повторите padded-value perturbation и объясните изменение результата.
2. Найдите в исходнике поведение при одном и при нескольких prompts: packed и padded shapes.
3. Почему `eval()` само по себе не замораживает веса? Покажите разницу с `requires_grad_(False)`.

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы отличаете frozen text encoder от trainable text-refiner внутри DiT. Можете указать,
какая маска требуется для прямоугольного batch и какие части text conditioning зависят от diffusion time.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 01. K6 Lite: вся система и training map](01_system_and_training_map.ipynb) · [03. Видео VAE: posterior, compression, causality и scaling →](03_video_vae.ipynb)
